In [1]:
import numpy as np
from tqdm.notebook import tqdm
from random import choice
from time import time
from matplotlib import pyplot as plt
from matplotlib import patches
import ipywidgets as widgets
from IPython.display import display
from time import time

UNKNOWN = 0
MISS = 1
HIT = 10
SUNK = 2


In [2]:
SIZE = 10
grid = np.ones((SIZE,SIZE)) * UNKNOWN
boats = {2:2, 3:1, 4:1, 5:1}

def check_available(grid, start_i, start_j, end_i, end_j):
    if end_i > grid.shape[0] or end_j> grid.shape[1]:
        return False
    return not grid[start_i:end_i, start_j: end_j].any()

def set_boat(grid, start_i, start_j, end_i, end_j):
    start_i_dil = max(0, start_i - 1)
    start_j_dil = max(0, start_j - 1)
    end_i_dil = min(SIZE, end_i + 1)
    end_j_dil = min(SIZE, end_j + 1)
    grid[start_i_dil : end_i_dil, start_j_dil: end_j_dil] = MISS
    grid[start_i : end_i, start_j : end_j] = HIT
    

def solve_step(grid, boats, start_i, start_j, root = False):
    global layouts
        
    boats = {key:value for key,value in boats.items() if value > 0}
    
    if len(boats) == 0:
        layouts += grid
        return   
        
    boat_size = max(boats.keys())
    if root > 0:
        i_range = tqdm(range(start_i, SIZE), leave = False, postfix = f"{root} i")
    else:
        i_range = range(start_i, SIZE)
    
    for i in i_range:
        if root > 0:
            j_range = tqdm(range(SIZE), leave = False, postfix = f"{root} j")
        else:
            j_range = range(SIZE)
        for j in j_range:
            if i == start_i and j < start_j:
                continue  
            #horizontal check
            if check_available(grid, i, j, i + 1, j + boat_size):
                grid_move = grid.copy()
                set_boat(grid_move, i, j, i + 1, j + boat_size)
                if boats[boat_size] == 1:
                    boats.pop(boat_size)
                    solve_step(grid_move, boats, 0, 0, root - 1)
                    boats[boat_size] = 1
                else:
                    boats[boat_size] -= 1
                    solve_step(grid_move, boats, i, j + 1, root - 1)
                    boats[boat_size] += 1
            #vertical check
            if check_available(grid, i, j, i + boat_size, j + 1):
                grid_move = grid.copy()
                set_boat(grid_move, i, j, i + boat_size, j + 1)
                if boats[boat_size] == 1:
                    boats.pop(boat_size)
                    solve_step(grid_move, boats, 0, 0, root - 1)
                    boats[boat_size] = 1
                else:
                    boats[boat_size] -= 1
                    solve_step(grid_move, boats, i, j + 1, root - 1)
                    boats[boat_size] += 1
            
    
    

def generate_uniform_layouts(size):
    layouts = {}
    for boat_size in range(2, 6):
        boat_layouts = []
        #horizontal
        for i in range(size):
            for j in range(size + 1 - boat_size):
                i_1 = max(0, i-1)
                j_1 = max(0, j-1)
                layout = np.zeros((size, size))
                layout[i_1:i+2, j_1:j+boat_size+1] = MISS
                layout[i:i+1,j:j+boat_size] = HIT
                boat_layouts.append(layout)
        for i in range(size + 1 - boat_size):
            for j in range(size):
                i_1 = max(0, i-1)
                j_1 = max(0, j-1)
                layout = np.zeros((size, size))
                layout[i_1:i+1+boat_size, j_1:j+2] = MISS
                layout[i:i+boat_size,j:j+1] = HIT
                boat_layouts.append(layout)
        layouts[boat_size] = boat_layouts
    return layouts

    
class Solver:
    def __init__(self, grid, boats):
        self.boats = boats
        self.layouts = generate_uniform_layouts(grid.shape[0])
        
        if grid is not None:
            misses = (grid == MISS) + (grid == SUNK)   
            for key in self.layouts:
                self.layouts[key] = [layout for layout in self.layouts[key] if not ((layout == HIT) & misses).any()]
        self.grid = grid.copy()
        self.hits = self.grid == HIT
        self.grid -= self.hits * HIT
        self.heatmap = np.zeros(grid.shape)
        self.n_solutions = 0

    def solve(self):
        
        self.step(self.grid, root = 2)

    def step(self, grid, root = 0):

        if len(self.boats) == 0:
            if (grid == HIT)[self.hits].all():
                self.heatmap += (grid==HIT) * (~self.hits)
                self.n_solutions += 1
            return
        
        boat_size = max(self.boats.keys())

        if root > 0:
            pbar = tqdm(self.layouts[boat_size], leave = False)
        else:
            pbar = self.layouts[boat_size]
        

        for layout in pbar:
            grid += layout
            if not (grid > HIT).any():
                if self.boats[boat_size] == 1:
                    boats.pop(boat_size)
                    self.step(grid, root - 1)
                    self.boats[boat_size] = 1
                else:
                    self.boats[boat_size] -= 1
                    self.step(grid, root - 1)
                    self.boats[boat_size] += 1
    
            grid -= layout
                
            

            

                    
        
        

def montecarlo(size, boats, existing_grid = None, attempts = 1000000000, successes = np.inf, callback = None):
    # hd = heatmap_display()
    layouts = generate_uniform_layouts(size)
    if existing_grid is not None:
        misses = (existing_grid == MISS) + (existing_grid == SUNK)
        hits = (existing_grid == HIT)
        
        for key in layouts:
            temp_layouts = [layout for layout in layouts[key] if not ((layout == HIT) & misses).any()]
            layouts[key] = temp_layouts
    else:
        hits = np.zeros((size, size))
    attempt = 0
    success = 0
    heatmap = np.zeros((size, size))
    pbar = tqdm(total = 100, mininterval = 0.02)
    # pbar = tqdm(range(attempts), mininterval = 0.02)
    sorted_boats = list(sorted(boats.items(), key = lambda x: -x[0]))
    
    for attempt in range(attempts):
        if success >= successes:
            break
        attempt += 1
        pbar.n = int(max((success / successes), attempt / attempts) * 100)
        pbar.update(0)
        layout = np.zeros((size, size))
        for key, value in sorted_boats:
            for _ in range(value):
                layout += choice(layouts[key])
                if (layout > HIT).any():
                    break
            if (layout > HIT).any():
                break
        if (layout > HIT).any():
            continue
        if (layout == HIT)[hits].all():
            success += 1
            heatmap += (layout == HIT) * (~hits)
            # pbar.set_description(f"{success/attempt*100:.1f}% success rate, {success} successful samples", refresh = False)
        if callback is not None:
            callback(success, attempt, heatmap)
    pbar.n = round(max((success / successes), attempt / attempts) * 100, 1)
    pbar.update(0)
    pbar.close()
    # hd.display(heatmap/success, force = True)
    # plt.close(hd.fig)
    return success, attempt, heatmap


class heatmap_display():
    FPS = 0.2
    def __init__(self):
        self.last_print = time()
        self.fig, self.axis = plt.subplots()
        self.axis.imshow(np.zeros((10,10)), cmap = 'gray')
        self.hfig = display(self.fig, display_id = True) 
        self.hfig.update(self.fig)
        self.artists = []
        

    def display(self, heatmap, force = False):
        if force or (time() - self.last_print > 1 / self.FPS):
            self.last_print = time()
            for artist in self.artists:
                artist.remove()
            self.artists = []            
            self.artists.append(self.axis.imshow(heatmap, cmap = 'gray', vmin = 0))
            max_i, max_j = np.where(heatmap == heatmap.max())
            for n in range(len(max_i)):
                i = max_i[n]
                j = max_j[n]
                rect = patches.Rectangle((j-0.5,i-0.5), 1, 1, facecolor = 'lightgreen', edgecolor = 'black', alpha = 0.3)
                self.artists.append(self.axis.add_patch(rect))
            self.fig.canvas.draw()
            self.fig.canvas.flush_events()
            self.hfig.update(self.fig)
                
                


def apply_hit(grid, i, j):
    grid[i,j] = HIT
    if i > 0:
        if j > 0:
            grid[i-1,j-1] = MISS
        if j + 1 < size:
            grid[i-1, j+1] = MISS
    if i + 1 < size:
        if j > 0:
            grid[i+1,j-1] = MISS
        if j + 1 < size:
            grid[i+1, j+1] = MISS

            
VERT = 1
HOR = 0

def sink_boat(grid, boats, i, j, d, length):
    if d == VERT:
        end_i = i + length
        end_j = j + 1
    else:
        end_i = i + 1
        end_j = j + length
    start_i_dil = max(0, i - 1)
    start_j_dil = max(0, j - 1)
    end_i_dil = min(SIZE, end_i + 1)
    end_j_dil = min(SIZE, end_j + 1)
    
    grid[start_i_dil : end_i_dil, start_j_dil: end_j_dil] = MISS
    grid[i : end_i, j : end_j] = SUNK

    boats[length] -= 1
    if boats[length] == 0:
        boats.pop(length)
    
    

In [3]:
zone_data = {
    1:  {"size": 3,  "boats": {3:1}},
    2:  {"size": 4,  "boats": {2:2}},
    3:  {"size": 5,  "boats": {2:2, 3:1}},
    4:  {"size": 6,  "boats": {2:1, 3:1, 4:1}},
    5:  {"size": 7,  "boats": {2:1, 3:2, 4:1}},
    6:  {"size": 8,  "boats": {2:2, 3:2, 5:1}},
    7:  {"size": 9,  "boats": {2:2, 3:2, 4:1, 5:1}},
    8:  {"size": 10, "boats": {2:2, 3:2, 4:2, 5:1}},
    9:  {"size": 10, "boats": {2:1, 3:2, 4:2, 5:1}},
    10: {"size": 10, "boats": {2:2, 3:1, 4:2, 5:1}},
    11: {"size": 10, "boats": {2:2, 3:1, 4:1, 5:1}}
}

In [4]:
class SonarOpsUI:
    def __init__(self):
        """
        simulate_fn(state) -> (r, c) or heatmap
        solve_fn(state) -> heatmap
        """
        self.UNKNOWN = 0
        self.HIT = 10
        self.MISS = 1
        self.SUNK = 2
        

        self.zone_dropdown = widgets.Dropdown(
            options=[(f"Zone {i}", i) for i in range(1, 12)],
            value=1,
            description='Zone:'
        )
        self.size = self.zone_to_size(self.zone_dropdown.value)
        self.sim_button = widgets.Button(description = "Simulate")
        self.sim_button.value = 1000
        # self.sim_buttons = []
        # for i in [100, 1000, 10000]:            
        #     button = widgets.Button(description=f"Simulate {i}")
        #     button.value = i
        #     self.sim_buttons.append(button)

        self.sunk_button = widgets.Button(description="Update Sunk Ships")

        self.grid = None
        self.zone_dropdown.observe(self.on_zone_change, names='value')
        # for button in self.sim_buttons:
        #     button.on_click(self.on_simulate)
        self.sim_button.on_click(self.on_simulate)
        self.sunk_button.on_click(self.update_sunk)

        self.container = widgets.VBox()
        self.build_ui()
        size = self.zone_to_size(self.zone_dropdown.value)
        self.heatmap = np.zeros((size, size))
        self.samples = 0
        self.fps = 5
        self.last_display = time()
        self.sunk_boats = []
        
    def zone_to_size(self, zone):
        return zone + 2 if zone <= 8 else 10

    def build_grid(self, size):
        self.buttons = []
        grid_items = []

        for r in range(size):
            row = []
            for c in range(size):
                btn = widgets.Button(
                    description='',
                    layout=widgets.Layout(width='60px', height='60px', padding='0px')
                )
                btn.state = self.UNKNOWN
                btn.style.button_color = 'lightgray'

                def on_click(b, r=r, c=c):
                    self.toggle_cell(b)

                btn.on_click(on_click)
                row.append(btn)
                grid_items.append(btn)

            self.buttons.append(row)

        self.grid = widgets.GridBox(
            grid_items,
            layout=widgets.Layout(
                grid_template_columns=f"repeat({size}, 60px)"
            )
        )

    def toggle_cell(self, btn):
        self.heatmap *= 0
        self.samples *= 0
        
        if btn.state == self.UNKNOWN:
            btn.state = self.HIT
            btn.style.button_color = 'palegreen'
        elif btn.state == self.HIT:
            btn.state = self.MISS
            btn.style.button_color = 'lightblue'
        else:
            btn.state = self.UNKNOWN
            btn.style.button_color = 'lightgrey'

    def get_state(self):
        return np.array([[btn.state for btn in row] for row in self.buttons])

    def update_sunk(self, _):
        np_state = self.get_state()
        np_state = np.pad(np_state, pad_width = 1, mode = 'constant', constant_values = self.MISS)
        sunk_templates = [np.array([self.MISS] + [self.HIT] * i + [self.MISS]) for i in range(2, 5)]
        sunk_templates.append(np.array([self.HIT]*5))

        for i in range(self.size + 2):
            for j in range(self.size + 2):
                if np_state[i,j] in (self.HIT, self.SUNK):
                    np_state[i+1,j+1] = self.MISS
                    np_state[i+1,j-1] = self.MISS
                    np_state[i-1,j+1] = self.MISS
                    np_state[i-1,j-1] = self.MISS
        
        for template in sunk_templates:
            template_length = len(template)
            template = template.reshape(-1, 1)
            for i in range(self.size + 3 - template_length):
                for j in range(1, self.size + 1):
                    if (template == np_state[i:i+template_length, j:j+1]).all():
                        sunk_template = template.copy()
                        sunk_template[sunk_template == self.HIT] = self.SUNK
                        np_state[i:i+template_length, j:j+1] = sunk_template
                        
                        
            template = template.reshape(1,-1)
            for i in range(1, self.size + 1):
                for j in range(self.size + 3 - template_length):
                    if (template == np_state[i:i+1, j:j+template_length]).all():
                        sunk_template = template.copy()
                        sunk_template[sunk_template == self.HIT] = self.SUNK
                        np_state[i:i+1, j:j+template_length] = sunk_template

        for i in range(self.size + 2):
            for j in range(self.size + 2):
                if np_state[i,j] == self.SUNK:
                    if np_state[i, j+1] not in (self.HIT, self.SUNK):
                        np_state[i,j+1] = self.MISS
                    if np_state[i, j-1] not in (self.HIT, self.SUNK):
                        np_state[i,j-1] = self.MISS
                    if np_state[i+1, j] not in (self.HIT, self.SUNK):
                        np_state[i+1,j] = self.MISS
                    if np_state[i-1, j] not in (self.HIT, self.SUNK):
                        np_state[i-1,j] = self.MISS
           
        np_state = np_state[1:-1,1:-1]

                
        for i in range(self.size):
            for j in range(self.size):
                if self.buttons[i][j].state != np_state[i,j]:
                    self.buttons[i][j].state = np_state[i,j]
                    if np_state[i,j] == self.HIT:
                        self.buttons[i][j].style.button_color = 'palegreen'
                    elif np_state[i,j] == self.MISS:
                        self.buttons[i][j].style.button_color = 'lightblue'
                    elif np_state[i,j] == self.SUNK:
                        self.buttons[i][j].style.button_color = 'green'
                    elif np_state[i,j] == self.UNKNOWN:
                        self.buttons[i][j].style.button_color = 'lightgrey'
                
        

    def render_heatmap(self, probs):    
        max_p = probs.max()
        if max_p == 0:
            max_p = 1.0

        norm = probs / max_p  
        best_idx = np.unravel_index(np.argmax(probs), probs.shape)
    
        rows, cols = probs.shape
        for r in range(rows):
            for c in range(cols):
                if self.buttons[r][c].state in [self.HIT, self.MISS, self.SUNK]:
                    self.buttons[r][c].description = " "
                    continue
                if (r, c) == best_idx:
                    color = "turquoise"
                else:
                    gray = int(255 * norm[r, c])
                    color = f"rgb({gray},{gray},{gray})"
    
                self.buttons[r][c].style.button_color = color
                self.buttons[r][c].description = f"{100 * probs[r, c]:.1f}%"

    def sunk_ships(self):
        state = self.get_state()
        sunk = {}
        for r in range(self.size):
            for c in range(self.size):
                if state[r][c] == self.SUNK:
                    if c == 0 or state[r][c-1] != self.SUNK:
                        boat_size = 1
                        while c + boat_size < self.size:
                            if state[r][c+boat_size] == self.SUNK:
                                boat_size += 1
                            else:
                                break
                        if boat_size > 1:
                            sunk[boat_size] = sunk.get(boat_size, 0) + 1
                            
                    if r == 0 or state[r-1][c] != self.SUNK:
                        boat_size = 1
                        while r + boat_size < self.size:
                            if state[r+boat_size][c] == self.SUNK:
                                boat_size += 1
                            else:
                                break
                        if boat_size > 1:
                            sunk[boat_size] = sunk.get(boat_size, 0) + 1
        return sunk        
                        

    def on_simulate(self, button):
        successes = button.value
        zd = zone_data[self.zone_dropdown.value]
        sunk_ships = self.sunk_ships()

        boats = zd['boats'].copy()
        for key in boats:
            boats[key] -= sunk_ships.get(key, 0)
        grid = self.get_state()
        attempts = successes * 1000
        self.output.clear_output(wait = True)
        with self.output:
            print('sunk ships', sunk_ships)
            print('ships remaining', boats)
        with self.output:       
            success, attempt, heatmap = montecarlo(zd["size"], boats, existing_grid = grid, attempts = attempts, successes = successes)        
        self.heatmap += heatmap
        self.samples += success        
        self.update_display(0, 0, None)
        
            
    def update_display(self, success, attempt, heatmap):
        if self.samples + success == 0:
            return
        if time() - self.last_display < 1/self.fps:
            return
        self.last_display = time()
        if heatmap is not None:
            heatmap = self.heatmap + heatmap
        else:
            heatmap = self.heatmap
        samples = self.samples + success
        self.render_heatmap(heatmap / samples)
        p = heatmap.max() / samples
        error = (p * (1-p) / samples)**0.5 * 100
        self.title.clear_output(wait = True)
        with self.title:
            print(f"Number of simulations: {samples}. Standard error for probabilities: ~{error:.2f}%")

        

    def on_zone_change(self, change):        
        self.size = self.zone_to_size(self.zone_dropdown.value)
        self.heatmap = np.zeros((self.size, self.size))
        self.samples = 0
        self.build_ui()
        

    def build_ui(self):
        self.build_grid(self.size)
        self.output = widgets.Output()
        self.title = widgets.Output()

        self.container.children = [self.zone_dropdown, self.title, self.grid, widgets.HBox([self.sim_button, self.sunk_button]), self.output]

    def display(self):
        display(self.container)



def dummy_sim(state):
    return (0, 0)


def dummy_solve(state):
    size = len(state)
    import random
    return [[random.random() for _ in range(size)] for _ in range(size)]

In [5]:
ui = SonarOpsUI()
ui.display()


# Sonar Ops Simulator

Just put in the state of your sonar-ops board and this simulator will tell you which cells are most likely hits.

#### What do the colours mean?

```
     Green = hit
      Blue = miss
Dark Green = sunk
 Turquoise = Highest Likelihood of Hit
      Grey = Greyscale heatmap of hit likelihood
```

#### How do I make a ship appear sunk?
At the moment, the best way is to change the state of the cells on either end to sink, and then press the "update Sunk Ships" icon.

#### What does "simulate" do?

The program samples from all possible orientations uniformly 1000 times and reports on how often each cell was a hit.

#### Can I run it for more than 1000 Simulations?

Yes! Press the simulate button again and it will combine the results.

#### Are there any known issues?

At the moment, the solver does not know that ships that aren't 'sunk' must be extended. So you may see times when the result should say "100%" when it does not.